# M3L4 E03 — Debugging con traces
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

**Objetivo:** aprender a leer trazas para detectar y diagnosticar fallas típicas en sistemas de agentes.

## Fallas típicas en sistemas de agentes

| # | Tipo de falla | Síntoma en la traza |
|---|---|---|
| 1 | **Misclassification** | `expected_intent ≠ actual_intent` en metadata |
| 2 | **Retrieval vacío** | Span de retrieval con `output: []` |
| 3 | **Latencia alta** | `duration_ms > 3000` en algún span |
| 4 | **Loop de agentes** | Mismo nodo aparece más de 2 veces en spans |
| 5 | **Error silencioso** | Span con `output: null` sin error marcado |

In [ ]:
import json

## Caso 1 — Misclassification

In [ ]:
trace_misclassification = {
    'trace_name': 'support-request',
    'input': {'query': 'No puedo ver mi factura'},
    'metadata': {
        'expected_intent': 'finance',
        'actual_intent': 'it'
    },
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'No puedo ver mi factura'},
            'output': {'intent': 'it'},
            'duration_ms': 140
        },
        {
            'name': 'it-agent',
            'input': {'query': 'No puedo ver mi factura'},
            'output': {'response': 'Probá reiniciar la app.'},
            'duration_ms': 900
        }
    ],
    'output': {'final_response': 'Probá reiniciar la app.'}
}

trace_misclassification

## Caso 2 — Retrieval vacío

In [ ]:
trace_empty_retrieval = {
    'trace_name': 'rag-support-request',
    'input': {'query': '¿Cuál es la política de licencia por maternidad?'},
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': '¿Cuál es la política de licencia por maternidad?'},
            'output': {'intent': 'hr'},
            'duration_ms': 110
        },
        {
            'name': 'retrieval',
            'input': {'query': '¿Cuál es la política de licencia por maternidad?'},
            'output': {'documents': [], 'count': 0},
            'duration_ms': 250
        },
        {
            'name': 'hr-agent',
            'input': {'query': '¿Cuál es la política de licencia por maternidad?', 'context': ''},
            'output': {'response': 'No tengo información disponible sobre ese tema.'},
            'duration_ms': 820
        }
    ],
    'output': {'final_response': 'No tengo información disponible sobre ese tema.'}
}

trace_empty_retrieval

## Caso 3 — Latencia alta

In [ ]:
trace_high_latency = {
    'trace_name': 'support-request',
    'input': {'query': '¿Cómo solicito vacaciones?'},
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': '¿Cómo solicito vacaciones?'},
            'output': {'intent': 'hr'},
            'duration_ms': 95
        },
        {
            'name': 'hr-agent',
            'input': {'query': '¿Cómo solicito vacaciones?'},
            'output': {'response': 'Para solicitar vacaciones ingresá al portal de RRHH.'},
            'duration_ms': 5800  # <-- alto!
        }
    ],
    'output': {'final_response': 'Para solicitar vacaciones ingresá al portal de RRHH.'}
}

trace_high_latency

## Caso 4 — Loop de agentes

In [ ]:
trace_loop = {
    'trace_name': 'support-request',
    'input': {'query': 'Mi laptop está lenta y tengo un problema con mi factura'},
    'spans': [
        {'name': 'supervisor', 'output': {'next': 'it-agent'}, 'duration_ms': 130},
        {'name': 'it-agent', 'output': {'response': 'Revisá tu conexión', 'handoff': 'finance-agent'}, 'duration_ms': 720},
        {'name': 'finance-agent', 'output': {'response': 'Revisá tu factura', 'handoff': 'it-agent'}, 'duration_ms': 680},
        {'name': 'it-agent', 'output': {'response': 'Revisá tu conexión', 'handoff': 'finance-agent'}, 'duration_ms': 710},
        {'name': 'finance-agent', 'output': {'response': 'Revisá tu factura', 'handoff': None}, 'duration_ms': 690}
    ],
    'output': {'final_response': 'Revisá tu factura'}
}

trace_loop

## Caso 5 — Error silencioso

In [ ]:
trace_silent_error = {
    'trace_name': 'support-request',
    'input': {'query': 'Necesito ver mi contrato'},
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'Necesito ver mi contrato'},
            'output': {'intent': 'legal'},
            'duration_ms': 100
        },
        {
            'name': 'legal-agent',
            'input': {'query': 'Necesito ver mi contrato'},
            'output': None,  # <-- sin output y sin error marcado
            'duration_ms': 15
        }
    ],
    'output': {'final_response': None}
}

trace_silent_error

## TODO — Función `diagnose_trace`

Implementá la función que detecta automáticamente el tipo de problema en una traza.

In [ ]:
def diagnose_trace(trace: dict) -> dict:
    """
    Analiza una traza y devuelve el diagnóstico del problema encontrado.

    Debe detectar (en orden):
    1. misclassification: metadata.expected_intent != metadata.actual_intent
    2. empty_retrieval: algún span de retrieval con output.count == 0
    3. high_latency: algún span con duration_ms > 3000
    4. agent_loop: algún nodo aparece más de 2 veces en los spans
    5. silent_error: algún span con output == None

    Returns:
        dict con 'problem', 'details' y 'suggested_fix'
    """
    # TODO 1: detectar misclassification
    # Pista: comparar metadata.get('expected_intent') con metadata.get('actual_intent')

    # TODO 2: detectar retrieval vacío
    # Pista: buscar span con 'retrieval' en el nombre y output.get('count') == 0

    # TODO 3: detectar latencia alta
    # Pista: iterar spans y verificar duration_ms > 3000

    # TODO 4: detectar loop de agentes
    # Pista: Counter de span names, buscar si alguno aparece > 2 veces

    # TODO 5: detectar error silencioso
    # Pista: buscar span con output == None

    return {'problem': 'none', 'details': 'No se detectaron problemas.', 'suggested_fix': None}

print('Función definida.')

## Ejecutar el diagnóstico en todos los casos

In [ ]:
cases = [
    ('Misclassification', trace_misclassification),
    ('Retrieval vacío',   trace_empty_retrieval),
    ('Latencia alta',     trace_high_latency),
    ('Loop de agentes',  trace_loop),
    ('Error silencioso',  trace_silent_error)
]

for name, trace in cases:
    result = diagnose_trace(trace)
    print(f'--- {name} ---')
    print(f"  Problema:   {result.get('problem')}")
    print(f"  Detalles:   {result.get('details')}")
    print(f"  Fix sugerido: {result.get('suggested_fix')}")
    print()

In [ ]:
assert diagnose_trace(trace_misclassification)['problem'] == 'misclassification'
assert diagnose_trace(trace_empty_retrieval)['problem'] == 'empty_retrieval'
assert diagnose_trace(trace_high_latency)['problem'] == 'high_latency'
assert diagnose_trace(trace_loop)['problem'] == 'agent_loop'
assert diagnose_trace(trace_silent_error)['problem'] == 'silent_error'
print('Checks E03 OK ✅')